In [ ]:
%load_ext autoreload
%autoreload 2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import h5py
import sys
sys.path.insert(0, '/Users/smericks/Desktop/StrongLensing/padma_timedelays/fasttdc/')
import Modeling.MassModels.paltas_preds as paltas_preds

Step #0: produce the approximate image models from NPE and their associated paltas-formatted metadata_truth.csv

In [ ]:
# step 1: convert time-delay catalog to paltas format
multiband_df = pd.read_csv('DataVectors/deblended_time_delays.csv')
for key in multiband_df.keys():
    print(key)

In [ ]:
# WE'RE USING i-BAND FOR EVERYTHING!!

key_translator = {
    # LENS MASS
    'main_deflector_parameters_theta_E': 'deflector_mass_theta_E',
    'main_deflector_parameters_gamma1': 'deflector_mass_gamma1',
    'main_deflector_parameters_gamma2': 'deflector_mass_gamma2',
    'main_deflector_parameters_gamma': 'deflector_mass_gamma',
    'main_deflector_parameters_e1': 'deflector_mass_e1',
    'main_deflector_parameters_e2': 'deflector_mass_e2',
    'main_deflector_parameters_center_x': 'deflector_mass_center_x',
    'main_deflector_parameters_center_y': 'deflector_mass_center_y',
    'main_deflector_parameters_z_lens': 'deflector_redshift',


    # LENS LIGHT
    'lens_light_parameters_R_sersic': 'deflector_light_i_R_sersic',
    'lens_light_parameters_center_x': 'deflector_light_i_center_x',
    'lens_light_parameters_center_y': 'deflector_light_i_center_y',
    'lens_light_parameters_e1': 'deflector_light_i_e1',
    'lens_light_parameters_e2': 'deflector_light_i_e2',
    'lens_light_parameters_mag_app': 'deflector_light_i_magnitude',
    'lens_light_parameters_n_sersic': 'deflector_light_i_n_sersic',
    'lens_light_parameters_z_source': 'deflector_redshift',


    # SOURCE
    'source_parameters_R_sersic': 'extended_source_light_i_R_sersic',
    'source_parameters_center_x': 'extended_source_light_i_center_x',
    'source_parameters_center_y': 'extended_source_light_i_center_y',
    'source_parameters_e1': 'extended_source_light_i_e1',
    'source_parameters_e2': 'extended_source_light_i_e2',
    'source_parameters_mag_app': 'extended_source_light_i_magnitude',
    'source_parameters_n_sersic': 'extended_source_light_i_n_sersic',
    'source_parameters_z_source': 'source_redshift',

    # POINT SOURCE
    'point_source_parameters_mag_app': 'ps_i_mag_true',
    'point_source_parameters_x_point_source': 'extended_source_light_i_center_x',
    'point_source_parameters_y_point_source': 'extended_source_light_i_center_y',
    'point_source_parameters_z_point_source': 'source_redshift',
}

paltas_df = pd.DataFrame()
paltas_df['catalog_idx'] = multiband_df['dataset']
for key in key_translator.keys():
    paltas_df[key] = multiband_df[key_translator[key]]

In [ ]:
paltas_df

In [ ]:
# step 2: construct HST-quality paltas network predictor
hst_preds = paltas_preds.PaltasPreds('/Users/smericks/Desktop/StrongLensing/darkenergy-from-LAGN/Modeling/MassModels/hst_training_config.py',
    numpix=165,
    model_weights='/Users/smericks/Desktop/StrongLensing/darkenergy-from-LAGN/Modeling/MassModels/xresnet34_hst_epoch72.h5',
    model_norms='/Users/smericks/Desktop/StrongLensing/darkenergy-from-LAGN/Modeling/MassModels/hst_norms.csv')

In [ ]:
# step 3 produce predicted image models
images_hst, metadata_list_hst, y_pred_hst, std_pred_hst, cov_pred_hst = hst_preds.preds_from_params(paltas_df)

In [ ]:
multiband_df.shape

In [ ]:
def save_h5(h5_path,catalog_idxs,images,mu_npe,cov_npe):
    h5f = h5py.File(h5_path, 'w')
    h5f.create_dataset('catalog_idx', data=catalog_idxs)
    h5f.create_dataset('images_array', data=images)
    h5f.create_dataset('mu_npe',data=mu_npe)
    h5f.create_dataset('cov_npe',data=cov_npe)
    h5f.close()

# save full paltas formatted metadata with catalog_idx
metadata_df_hst = pd.DataFrame(metadata_list_hst)
metadata_df_hst['catalog_idx'] = paltas_df['catalog_idx']


# GOLD
metadata_df_hst.to_csv('DataVectors/truth_metadata.csv', index=False)
save_h5('DataVectors/hst_image_models.h5',
        catalog_idxs=metadata_df_hst.loc[:,'catalog_idx'].to_numpy(),
        images=images_hst,
        mu_npe=y_pred_hst,
        cov_npe=cov_pred_hst)